# Taiga-tundra ecotone (TTE) map on MAAP DPS

Runs `lib/tte_map.py` (Python port of the GEE TTE workflow, Montesano et al. 2020 ERL) as the DPS
algorithm `run_tte_map`, one or more 1-degree tiles per job. Each job writes:

* one 5-band COG per tile (`tte_<tcc label>_<tile_num>.tif`: tte_domain, tte_zones, tte_abruptness,
  tte_classification, tcc), and
* STAC metadata (`write_stac=true`): `catalog.json`, `tte_map/collection.json` and one item per tile.
  DPS publishes the items in the MAAP DPS STAC catalog, collection `{user}__run_tte_map__{version}`,
  where `notebooks/TTE_review_map.ipynb` maps them.

**What each STAC item records** (see `lib/tte_stac.py`):

| Group | Fields |
|---|---|
| Identity | id `tte_<tcc label>_<tile_num>_<parameter signature>`; footprint = the 1-degree tile; datetime = `tcc_year` (Jan 1) |
| How it was made | `tte:tcc_source`, `tte:tcc_label`, `tte:tcc_year`, `tte:tcc_description`, Hansen version or Meta CHM version and height threshold or input paths, `tte:water`, `tte:domain`, `tte:gee_pyramid_emulation`, `tte:zone_thresholds_pct`, `tte:code_version` (git tag), `tte:method`; DPS adds `maap-dps:*` (user, algorithm, job tag) and `created` |
| What is in the file | asset `tte` (COG), band names and descriptions, `raster:bands` (float32, nodata NaN, 30 m), `classification:classes` on band 4 (values 1-11, names, colors), `proj:*` (polar LAEA grid, WKT2) |
| Per-tile summary | `tte:domain_km2`, `tte:classified_km2`, `tte:classified_share_of_domain`, `tte:mean_tcc_domain`, `tte:mean_abruptness`, `tte:class_km2`, `tte:group_share` |
| Citation | `sci:doi`, `sci:citation` (Montesano et al. 2020); collection license CC-BY-4.0 |

Re-running a tile with the same parameters replaces its item; other parameters (tree cover source,
year, versions, options) give a new item in the same collection.

**Where to run:** MAAP Hub, a Python kernel; submitting needs your MAAP login, which the Hub kernel
has. Before registering, the code must be on GitHub with the tag named in `ALGO_VERSION`
(`github.com/pahbs/tte`).

**MAAP's OGC DPS (maap-py 5).** Algorithms are now OGC processes described by a CWL file; the
**Register Algorithm** form on MAAP Hub writes the CWL from the fields in section 1. Inputs reach
`run.sh` by name (`--tile_num 4209 --tcc_source hansen ...`), and jobs are submitted with
`maap.submit_job(process_id, inputs, queue, tag=...)`.

In [ ]:
import os, glob, json, datetime, warnings
warnings.filterwarnings('ignore', message='.*geographic CRS.*')
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from maap.maap import MAAP
maap = MAAP()

REPO = os.path.expanduser('~/code/tte')
ALG_YML = f'{REPO}/dps/alg_tte_map/run_tte_map.yml'     # the algorithm's fields and inputs (for the form)
ALGO_ID = 'run_tte_map'
ALGO_VERSION = 'tte_map_v1'          # git tag in github.com/pahbs/tte
USER = 'montesano'
QUEUE = 'maap-dps-worker-8gb'
COLLECTION = f'{USER}__{ALGO_ID}__{ALGO_VERSION}'.lower()   # where DPS publishes the STAC items
DPS_OUT = os.path.expanduser(f'~/my-private-bucket/dps_output/{ALGO_ID}/{ALGO_VERSION}')
print('STAC collection:', COLLECTION)

## 1. Register (once per version)

maap-py 5 has no `register_algorithm_from_yaml_file`. Use the **Register Algorithm** form on MAAP
Hub (MAAP menu or launcher) with the values printed below; it builds the container (running
`build.sh`) and the CWL. Wait for the build to finish (it shows in the Jobs view) before submitting.
Registering again with the same name and version replaces the algorithm.

In [ ]:
import yaml
cfg = yaml.safe_load(open(ALG_YML))
form = {
    'Algorithm Name': cfg['algorithm_name'],
    'Algorithm Version': cfg['algorithm_version'],
    'Algorithm Description': ' '.join(cfg['algorithm_description'].split()),
    'Repository URL': cfg['repository_url'],
    'Repository Branch': cfg['algorithm_version'] + '   (the git tag)',
    'Run Command': cfg['run_command'],
    'Build Command': cfg['build_command'],
    'Base Container URL': cfg['docker_container_url'],
    'Resource Requirements': f"{cfg['queue']}, disk {cfg['disk_space']}",
}
for k, v in form.items():
    print(f'{k:24s} {v}')
print('\nInputs (all type string; name / default / description):')
for i in cfg['inputs']['positional']:
    print(f"  {i['name']:16s} {str(i['default']):60s} {i['description'][:70]}")

After the build finishes: the OGC process id of the algorithm, needed to submit jobs.

In [ ]:
r = maap.list_algorithms(algorithm_name=ALGO_ID, algorithm_version=ALGO_VERSION)
listing = r.json() if hasattr(r, 'json') else r
procs = listing.get('processes', listing) if isinstance(listing, dict) else listing
print(json.dumps(procs, indent=1)[:1500])
PROCESS_ID = procs[0]['id'] if procs else None      # check this is run_tte_map / tte_map_v1
print('PROCESS_ID:', PROCESS_ID)

## 2. Inputs

* **`tcc_source='hansen'`** (default): Hansen GFC `treecover2000` minus loss before `tcc_year`.
  `GFC-2018-v1.6` and `tcc_year=2010` reproduce the 2010 map (the GEE original used calibrated 2 m TCC,
  `cal_2m_tcc`, in North America, so North America differs from TTE_DATA_v201902v2).
* **`tcc_source='meta_chm'`**: the latest Meta/WRI canopy height map (CHM v2): percent of ~1 m pixels
  >= 2 m in each 30 m cell. Set `tcc_year` to the nominal year of its imagery (dates vary by place); it is
  the item's datetime. About 15-30 min per tile (it reads ~0.5 m pixels).
* **`tcc_source='path'`**: any tree cover rasters in `tcc_paths` (comma-separated s3:// or https://;
  percent cover, or set `tcc_scale`), labelled by `out_name`.
* **`write_stac='true'`**: write the STAC metadata (keep it on).

`IDENTIFIER` is the DPS job tag: it groups jobs in `dps_output` and is `maap-dps:tag` in STAC.

In [ ]:
params = {
    'tile_num': '',
    'tcc_source': 'hansen',
    'tcc_year': '2010',
    'hansen_version': 'GFC-2018-v1.6',
    'tcc_paths': 'none',
    'tcc_scale': '1',
    'tcc_resampling': 'nearest',
    'out_name': 'none',
    'gee_emulation': 'true',
    'domain_path': 's3://maap-ops-workspace/shared/montesano/tte/static/tte_domain_v201902v2.vrt',
    'tiles_gpkg': 's3://maap-ops-workspace/shared/montesano/databank/tte_tiles.gpkg',
    'write_stac': 'true',
}
# params.update(tcc_source='meta_chm', tcc_year='2020')     # e.g. the Meta CHM v2 run

LABEL = {'hansen': f"hansen{params['tcc_year']}", 'meta_chm': 'metachm2m'}.get(params['tcc_source'], params['out_name'])
IDENTIFIER = f'tte_map_{LABEL}'

tiles = gpd.read_file(os.path.expanduser('~/my-public-bucket/databank/tte_tiles.gpkg')).to_crs(4326)
print(len(tiles), 'tiles in the TTE tile index;', 'job tag:', IDENTIFIER)


def _json(r):
    try:
        return r.json()
    except Exception:
        return {'http_status': getattr(r, 'status_code', None), 'text': getattr(r, 'text', str(r))[:300]}


def submit(tile_list, params, tag=IDENTIFIER):
    # one OGC job per tile; inputs by name (run.sh reads --name value)
    rows = []
    for tn in tile_list:
        r = maap.submit_job(PROCESS_ID, inputs=dict(params, tile_num=str(tn)), queue=QUEUE, tag=tag)
        j = _json(r)
        rows.append(dict(tile_num=tn, job_id=j.get('jobID') or j.get('id'), status=j.get('status'),
                         submit_time=datetime.datetime.now().isoformat(timespec='seconds'), response=j))
    return pd.DataFrame(rows)


def job_status(jobs):
    jobs = jobs.copy()
    jobs['status'] = [_json(maap.get_job_status(j)).get('status') for j in jobs.job_id]
    return jobs

## 3. Test one tile first

Check that the COG and the STAC files are written and that the item shows up in the DPS STAC catalog
before running anything at scale. Tile 4209 is the Kobuk-Noatak treeline in western Alaska.

In [ ]:
TEST_TILE = 4209
# jobs_test = submit([TEST_TILE], params); jobs_test

In [ ]:
# job_status(jobs_test)
# _json(maap.get_job_result(jobs_test.job_id[0]))      # output links of a finished job

**3a. The job's output folder:** the COG, `catalog.json`, `tte_map/collection.json` and
`tte_map/<item id>/<item id>.json`. `pystac` re-validates the catalog as DPS will read it.

In [ ]:
import pystac
cats = sorted(glob.glob(f'{DPS_OUT}/{IDENTIFIER}/**/catalog.json', recursive=True))
if not cats:   # the OGC DPS may lay out dps_output differently: look anywhere under this algorithm
    cats = sorted(glob.glob(os.path.expanduser(f'~/my-private-bucket/dps_output/{ALGO_ID}/**/catalog.json'), recursive=True),
                  key=os.path.getmtime)
print(len(cats), 'catalogs; latest:', cats[-1] if cats else None)
if cats:
    job_dir = os.path.dirname(cats[-1])
    for f in sorted(glob.glob(f'{job_dir}/**/*', recursive=True)):
        if os.path.isfile(f):
            print(' ', os.path.relpath(f, job_dir), f'{os.path.getsize(f) / 1e6:.1f} MB')
    cat = pystac.Catalog.from_file(cats[-1])
    cat.validate_all()
    for it in cat.get_items(recursive=True):
        print('valid item:', it.id, '->', it.assets['tte'].get_absolute_href())
        print({k: v for k, v in it.properties.items() if k.startswith('tte:') and not isinstance(v, dict)})

**3b. Published in the DPS STAC catalog?** Ingestion is asynchronous; give it a few minutes after the
job finishes. Browser: `https://dps-stac-browser.maap-project.org` (your catalog appears there after the
first job that writes STAC).

In [ ]:
from pystac_client import Client
stac = Client.open('https://dps-stac.maap-project.org')
search = stac.search(collections=[COLLECTION],
                     filter={'op': '=', 'args': [{'property': 'tte:tile_num'}, TEST_TILE]}, filter_lang='cql2-json')
found = list(search.items())
print(len(found), 'item(s) for tile', TEST_TILE, 'in', COLLECTION)
for it in found:
    print(it.id, '| tag:', it.properties.get('maap-dps:tag'), '| created:', it.properties.get('created'))
    print('  asset:', it.assets['tte'].href)
print(f'https://dps-stac-browser.maap-project.org/collections/{COLLECTION}')

## 4. Review set: western Alaska and the Brooks Range

The tiles of the TTE domain in two boxes, for `notebooks/TTE_review_map.ipynb`. Tiles outside the TTE
domain exit without output.

In [ ]:
REVIEW_BOXES = {'western Alaska': (-166.0, 63.5, -155.0, 69.0), 'Brooks Range': (-155.0, 66.5, -141.0, 69.0)}
review = pd.concat([tiles[tiles.intersection(box(*b)).area > 0.01].assign(region=name) for name, b in REVIEW_BOXES.items()])
review = review.drop_duplicates('tile_num')
REVIEW_TILES = review.tile_num.tolist()
print(review.groupby('region').size().to_dict(), '->', len(REVIEW_TILES), 'tiles')
review.plot(column='region', legend=True, figsize=(8, 4), edgecolor='k', lw=0.3);

In [ ]:
# jobs_review = submit(REVIEW_TILES, params)
# jobs_review.to_csv(os.path.expanduser(f'~/my-public-bucket/dps_submission_results/DPS_{ALGO_ID}_{IDENTIFIER}_review_{datetime.date.today()}.csv'), index=False)
# job_status(jobs_review).status.value_counts()

## 5. Compare with the GEE product (optional)
`validation/validate_tte_vs_gee.py` resamples a Python tile onto the matching GEE tile grid and reports agreement.

In [ ]:
# !python ~/code/tte/validation/validate_tte_vs_gee.py --tile_num 4209 --py_dir <job folder with the tif>

## 6. All tiles
Submit in batches. Hansen is ~1 min per tile on the 8 GB worker; Meta CHM 15-30 min.

In [ ]:
ALL_TILES = tiles.tile_num.tolist()
# jobs_all = submit(ALL_TILES, params)
# jobs_all.to_csv(os.path.expanduser(f'~/my-public-bucket/dps_submission_results/DPS_{ALGO_ID}_{IDENTIFIER}_{datetime.date.today()}.csv'), index=False)